# Bitcoin Fraud Detection Using Graph Neural Networks

## Table of contents
1. Introduction and research objectives
2. Business understanding
3. Dataset loading and exploration
4. Exploratory data analysis
5. Graph construction and visualization
6. Preprocessing and temporal splitting
7. GraphSAGE
8. GAT
9. Model comparison
10. Evaluation and reporting
11. Conclusion

This notebook compares GraphSAGE, GAT, and an MLP baseline on the original Elliptic Bitcoin Transaction Dataset. It uses a leakage-safe temporal evaluation protocol and reports only executed results.

### Execution modes
The notebook defaults to **Fast** mode. Change `FAST_MODE` in the setup cell to `False` for the fuller Normal run. Fast mode reduces epochs, patience, seeds, and optional visualizations; it keeps the temporal split, weighted loss, validation threshold selection, and test-only final evaluation unchanged.

## 1. Introduction and research objectives

**Goal.** Detect illicit Bitcoin transactions while comparing graph-based models with a non-graph baseline. The research questions are whether graph structure improves illicit detection, whether attention helps over mean aggregation, how weighted BCE affects minority-class performance, and how models behave under temporal drift.

The project follows CRISP-DM: business understanding, data understanding, preparation, modeling, evaluation, and reporting. The dataset is the Elliptic transaction graph from Weber et al. (2019). The main metric is illicit-class F1, complemented by precision, recall, ROC-AUC, PR-AUC, and confusion matrices.

In [1]:
# --- 1.1 Reproducible setup and execution mode ---
import os
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
VENV_DIR = Path.cwd() / ".venv"
if VENV_DIR.exists():
    sys.path.insert(0, str(VENV_DIR / "Lib/site-packages"))
    if os.name == "nt":
        os.add_dll_directory(str(VENV_DIR / "Scripts"))
import torch

warnings.filterwarnings("ignore", message=".*torch.jit.script.*", category=FutureWarning)

SEED = 42
FAST_MODE = False
NORMAL_MODE = not FAST_MODE
EPOCHS = 30 if FAST_MODE else 200
PATIENCE = 5 if FAST_MODE else 30
N_SEEDS = 1 if FAST_MODE else 5
TRAIN_STEPS, VAL_STEPS, TEST_STEPS = range(1, 30), range(30, 35), range(35, 50)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = SEED) -> None:
    """Make Python, NumPy, and PyTorch runs reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
ROOT = Path.cwd()
RAW_DIR, PROC_DIR = ROOT / "data/raw", ROOT / "data/processed"
FIG_DIR, TAB_DIR, MODEL_DIR = ROOT / "outputs/figures", ROOT / "outputs/tables", ROOT / "outputs/models"
for directory in (PROC_DIR, FIG_DIR, TAB_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
assert RAW_DIR.exists(), "Run the notebook from the project root."
print({"mode": "FAST" if FAST_MODE else "NORMAL", "epochs": EPOCHS, "patience": PATIENCE, "seeds": N_SEEDS, "device": str(DEVICE)})

{'mode': 'NORMAL', 'epochs': 200, 'patience': 30, 'seeds': 5, 'device': 'cpu'}


## 2. Business understanding

In anti-money-laundering work, a false negative can allow illicit activity to continue, while a false positive consumes investigator time. Transaction relationships can provide context that local transaction attributes cannot. Accuracy is misleading because illicit transactions are the minority class, so illicit F1 and PR-AUC are prioritized.

In [2]:
# --- 3.1 Load and align the Elliptic tables ---
feats = pd.read_csv(RAW_DIR / "elliptic_txs_features.csv", header=None)
feats.columns = (["txId", "time_step"] + [f"local_{i}" for i in range(1, 94)] + [f"agg_{i}" for i in range(1, 73)])
classes = pd.read_csv(RAW_DIR / "elliptic_txs_classes.csv")
edges = pd.read_csv(RAW_DIR / "elliptic_txs_edgelist.csv")
df = feats[["txId", "time_step"]].merge(classes, on="txId", how="left")
df["label"] = df["class"].astype(str).map({"1": 1, "2": 0, "unknown": -1})
assert df["label"].notna().all(), "Unmapped class values"
assert np.array_equal(df["txId"].to_numpy(), feats["txId"].to_numpy()), "Transaction order mismatch"
print("features", feats.shape, "classes", classes.shape, "edges", edges.shape)
print(df["label"].value_counts().rename({0: "licit", 1: "illicit", -1: "unknown"}).to_dict())
display(feats.head(3), classes.head(3), edges.head(3))

features (203769, 167) classes (203769, 2) edges (234355, 2)
{'unknown': 157205, 'licit': 42019, 'illicit': 4545}


,txId,time_step,local_1,local_2,local_3,local_4,local_5,local_6,local_7,local_8,...,agg_63,agg_64,agg_65,agg_66,agg_67,agg_68,agg_69,agg_70,agg_71,agg_72
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.12197,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.12197,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.12197,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870


## 3. Dataset loading and exploration

**Goal.** Verify table dimensions, label alignment, feature groups, time coverage, and edge validity before modeling.

In [3]:
# --- 3.2 Dataset integrity checks ---
feature_columns = [f"local_{i}" for i in range(1, 94)] + [f"agg_{i}" for i in range(1, 73)]
node_ids = set(df["txId"])
edge_source, edge_target = edges.columns[:2]
valid_edges = edges[edge_source].isin(node_ids) & edges[edge_target].isin(node_ids)
integrity = {
    "missing_values": int(feats.isna().sum().sum()),
    "duplicate_txIds": int(feats["txId"].duplicated().sum()),
    "invalid_edge_endpoints": int((~valid_edges).sum()),
    "self_loops": int((edges[edge_source] == edges[edge_target]).sum()),
    "duplicate_edges": int(edges.duplicated().sum()),
    "feature_count": len(feature_columns),
    "time_min": int(df.time_step.min()),
    "time_max": int(df.time_step.max()),
}
print(integrity)
assert integrity["invalid_edge_endpoints"] == 0
assert len(feature_columns) == 165

{'missing_values': 0, 'duplicate_txIds': 0, 'invalid_edge_endpoints': 0, 'self_loops': 0, 'duplicate_edges': 0, 'feature_count': 165, 'time_min': 1, 'time_max': 49}


## 4. Exploratory data analysis

**Goal.** Quantify class imbalance and temporal coverage without using test labels to make modeling choices.

In [4]:
# --- 4.1 Class and temporal summaries ---
label_names = {-1: "unknown", 0: "licit", 1: "illicit"}
label_summary = df["label"].map(label_names).value_counts().rename_axis("label").to_frame("count")
label_summary["percent_all"] = 100 * label_summary["count"] / len(df)
label_summary["percent_labelled"] = 100 * label_summary["count"] / label_summary.loc[["licit", "illicit"], "count"].sum()
display(label_summary)
temporal_summary = df.groupby("time_step").agg(nodes=("txId", "size"), illicit=("label", lambda x: int((x == 1).sum())), labelled=("label", lambda x: int((x != -1).sum())))
temporal_summary["illicit_rate_labelled"] = temporal_summary["illicit"] / temporal_summary["labelled"].replace(0, np.nan)
display(temporal_summary.head(), temporal_summary.tail())

,count,percent_all,percent_labelled
label,,,
unknown,157205,77.148634,337.610600
licit,42019,20.620899,90.239241
illicit,4545,2.230467,9.760759


,nodes,illicit,labelled,illicit_rate_labelled
time_step,,,,
1,7880,17,2147,0.007918
2,4544,18,1117,0.016115
3,6621,11,1279,0.008600
4,5693,30,1440,0.020833
5,6803,8,1882,0.004251


,nodes,illicit,labelled,illicit_rate_labelled
time_step,,,,
45,5598,5,1221,0.004095
46,3519,2,712,0.002809
47,5121,22,846,0.026005
48,2954,36,471,0.076433
49,2454,56,476,0.117647


## 5. Graph construction and visualization

**Goal.** Map transaction IDs to contiguous tensor indices and create an undirected message-passing graph. Edges are treated as undirected because the neighborhood context is used symmetrically for node classification.

In [5]:
# --- 5.1 Build the PyTorch Geometric graph ---
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

id_to_idx = pd.Series(np.arange(len(df)), index=df.txId).to_dict()
source_idx = edges[edge_source].map(id_to_idx).to_numpy()
target_idx = edges[edge_target].map(id_to_idx).to_numpy()
edge_index = torch.tensor(np.vstack([source_idx, target_idx]), dtype=torch.long)
edge_index = to_undirected(edge_index)
data = Data(
    x=torch.tensor(feats[feature_columns].to_numpy(dtype=np.float32)),
    edge_index=edge_index,
    y=torch.tensor(df.label.to_numpy(dtype=np.int64)),
    time_step=torch.tensor(df.time_step.to_numpy(dtype=np.int64)),
)
print(data)
print("undirected edges:", data.edge_index.shape[1])

Data(x=[203769, 165], edge_index=[2, 468710], y=[203769], time_step=[203769])
undirected edges: 468710


## 6. Preprocessing and temporal train / validation / test splitting

**Goal.** Fit preprocessing only on training time steps, exclude unknown labels from supervised loss and metrics, and keep each split temporally isolated.

In [6]:
# --- 6.1 Leakage-safe scaling and masks ---
from sklearn.preprocessing import StandardScaler
from torch_geometric.utils import subgraph

train_nodes = df.time_step.isin(TRAIN_STEPS).to_numpy()
scaler = StandardScaler().fit(feats.loc[train_nodes, feature_columns])
x_scaled = scaler.transform(feats[feature_columns]).astype(np.float32)
data.x = torch.tensor(x_scaled)
train_mask = torch.tensor(train_nodes & (df.label.to_numpy() != -1))
val_mask = torch.tensor(df.time_step.isin(VAL_STEPS).to_numpy() & (df.label.to_numpy() != -1))
test_mask = torch.tensor(df.time_step.isin(TEST_STEPS).to_numpy() & (df.label.to_numpy() != -1))
n_licit = int(((data.y == 0) & train_mask).sum())
n_illicit = int(((data.y == 1) & train_mask).sum())
POS_WEIGHT = torch.tensor([n_licit / n_illicit], dtype=torch.float32, device=DEVICE)
split_summary = pd.DataFrame([
    {"split": name, "nodes": int(mask.sum()), "illicit": int(((data.y == 1) & mask).sum()), "licit": int(((data.y == 0) & mask).sum())}
    for name, mask in (("train", train_mask), ("validation", val_mask), ("test", test_mask))
])
split_summary["illicit_rate"] = split_summary.illicit / split_summary[["illicit", "licit"]].sum(axis=1)
split_summary.to_csv(TAB_DIR / "split_summary.csv", index=False)
print("features:", data.x.shape[1], "pos_weight:", float(POS_WEIGHT), "device:", DEVICE)
display(split_summary)

features: 165 pos_weight: 8.1887845993042 device: cpu


,split,nodes,illicit,licit,illicit_rate
0,train,26381,2871,23510,0.108828
1,validation,3513,591,2922,0.168232
2,test,16670,1083,15587,0.064967


In [7]:
# --- 6.2 Shared evaluation and threshold utilities ---
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)

def evaluate(y_true, probabilities, threshold=0.5):
    """Calculate illicit-class metrics for labelled nodes."""
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities)
    predicted = (probabilities >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "f1": f1_score(y_true, predicted, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "confusion_matrix": confusion_matrix(y_true, predicted).tolist(),
    }

def best_threshold_on_val(y_true, probabilities):
    """Select the threshold with maximum illicit F1 on validation data."""
    thresholds = np.arange(0.05, 0.951, 0.01)
    scores = [f1_score(y_true, probabilities >= threshold, zero_division=0) for threshold in thresholds]
    return float(thresholds[int(np.argmax(scores))]), thresholds, np.asarray(scores)


## 7–10. Model training, comparison, and evaluation

The modelling phase trains the MLP baseline, GraphSAGE, and GAT with weighted cross-entropy. The implementation is shared with `train_models.py` so the notebook and command-line run produce the same artifacts. The validation set selects the illicit-class F1 threshold; the test set is evaluated only after that threshold is frozen.

In [8]:
# --- 7–10. Train models and display test comparison ---
from train_models import main as train_models_main

# Run all cells without retraining: display the already saved normal-mode results.
model_results = pd.read_csv(TAB_DIR / "model_results.csv")
display(model_results)

# Optional: run train_models_main(["--normal"]) only in this dedicated cell
# when a new 5-seed experiment is intentionally needed.

,model,best_epoch,epochs_run,validation_f1,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp,seed,device,fast_mode,train_nodes,validation_nodes,test_nodes
0,MLP,83,113,0.843034,0.90,0.446792,0.527239,0.483693,0.870318,0.313774,14880,707,512,571,42,cpu,False,26381,3513,16670
1,GraphSAGE,86,116,0.895623,0.90,0.665934,0.559557,0.608128,0.870095,0.545139,15283,304,477,606,42,cpu,False,26381,3513,16670
2,GAT,41,71,0.822934,0.93,0.414458,0.476454,0.443299,0.862399,0.381462,14858,729,567,516,42,cpu,False,26381,3513,16670
3,MLP,106,136,0.876923,0.89,0.453583,0.514312,0.482042,0.867546,0.315607,14916,671,526,557,43,cpu,False,26381,3513,16670
4,GraphSAGE,162,192,0.912106,0.90,0.497276,0.590028,0.539696,0.871642,0.472087,14941,646,444,639,43,cpu,False,26381,3513,16670
5,GAT,126,156,0.851101,0.95,0.508251,0.426593,0.463855,0.846176,0.427721,15140,447,621,462,43,cpu,False,26381,3513,16670
6,MLP,141,171,0.879252,0.87,0.444444,0.539243,0.487276,0.860098,0.298349,14857,730,499,584,44,cpu,False,26381,3513,16670
7,GraphSAGE,94,124,0.899317,0.95,0.503683,0.505078,0.504380,0.853599,0.462390,15048,539,536,547,44,cpu,False,26381,3513,16670
8,GAT,105,135,0.840198,0.87,0.427068,0.524469,0.470783,0.857584,0.389274,14825,762,515,568,44,cpu,False,26381,3513,16670
9,MLP,138,168,0.890940,0.71,0.430370,0.505078,0.464741,0.858729,0.291986,14863,724,536,547,45,cpu,False,26381,3513,16670
